### 1. Carga de Librerías y Dataset
Importamos las herramientas necesariass como `pgmpy` para redes bayesianas.

In [4]:
import sys
import subprocess

# Lista de librerías necesarias (incluimos openpyxl para poder leer archivos de Excel)
librerias_requeridas = ['pandas', 'pgmpy', 'openpyxl']

# Bucle para comprobar e instalar si es necesario
for lib in librerias_requeridas:
    try:
        __import__(lib)
    except ImportError:
        print(f"La librería '{lib}' no está instalada. Instalando ahora...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", lib])

La librería 'pandas' no está instalada. Instalando ahora...
La librería 'pgmpy' no está instalada. Instalando ahora...
La librería 'openpyxl' no está instalada. Instalando ahora...


### 2. Carga del Dataset de excel
Cargamos nuestro dataset de transacciones bancarias.

In [ ]:
import pandas as pd
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.inference import VariableElimination
print("Librerias importadas correctamente.")

df = pd.read_excel('dataset_fraude_bancario_bayesiano.xlsx')
display(df.head())


Librerias importadas correctamente.


,Monto_Alto,Ubicacion_Extraña,Hora_Inusual,Dispositivo_Nuevo,Frecuencia_Transacciones,Fraude
0,Sí,Sí,Sí,Sí,Alta,Sí
1,Sí,Sí,No,Sí,Alta,Sí
2,Sí,Sí,Sí,No,Alta,Sí
3,Sí,No,Sí,Sí,Alta,Sí
4,No,Sí,Sí,Sí,Alta,Sí


### 3. Construcción del Modelo Naive Bayes
Definimos la estructura de la red. Bajo el enfoque Naive Bayes, asumimos independencia condicional: la variable objetivo (`Fraude`) actúa como nodo padre que condiciona directamente a todas las características de la transacción.

In [ ]:
# 1. Estructura correcta de Naive Bayes
modelo = DiscreteBayesianNetwork([
    ('Fraude', 'Monto_Alto'),
    ('Fraude', 'Ubicacion_Extraña'),
    ('Fraude', 'Dispositivo_Nuevo'),
    ('Fraude', 'Hora_Inusual'),
    ('Fraude', 'Frecuencia_Transacciones')
])

# 2. Entrenamos el modelo
modelo.fit(df)

print("Modelo Naive Bayes entrenado correctamente con la estructura adecuada.")

Modelo Naive Bayes entrenado correctamente con la estructura adecuada.


### 3. Tablas de Probabilidad Condicional (CPDs)
Extraemos y mostramos las distribuciones de probabilidad que el modelo aprendió automáticamente a partir de las frecuencias del dataset.

In [8]:
for cpd in modelo.get_cpds():
    print(cpd)

+------------+----------+
| Fraude(No) | 0.535714 |
+------------+----------+
| Fraude(Sí) | 0.464286 |
+------------+----------+
+----------------+---------------------+--------------------+
| Fraude         | Fraude(No)          | Fraude(Sí)         |
+----------------+---------------------+--------------------+
| Monto_Alto(No) | 0.5666666666666667  | 0.3076923076923077 |
+----------------+---------------------+--------------------+
| Monto_Alto(Sí) | 0.43333333333333335 | 0.6923076923076923 |
+----------------+---------------------+--------------------+
+-----------------------+--------------------+---------------------+
| Fraude                | Fraude(No)         | Fraude(Sí)          |
+-----------------------+--------------------+---------------------+
| Ubicacion_Extraña(No) | 0.6666666666666666 | 0.19230769230769232 |
+-----------------------+--------------------+---------------------+
| Ubicacion_Extraña(Sí) | 0.3333333333333333 | 0.8076923076923077  |
+---------------------

### 4. Predicción con Nuevos Samples (Inferencia)
Utilizamos el método de Eliminación de Variables para predecir la probabilidad de fraude de una transacción completamente nueva.

In [11]:
inferencia = VariableElimination(modelo)

print("--- NUEVO SAMPLE 1 (CASO LEGAL) ---")
resultado_1 = inferencia.query(
    variables=['Fraude'],
    evidence={'Monto_Alto': 'No', 'Ubicacion_Extraña': 'No', 'Hora_Inusual': 'No', 
              'Dispositivo_Nuevo': 'No', 'Frecuencia_Transacciones': 'Media'}
)
print(resultado_1)

print("\n--- NUEVO SAMPLE 2 (CASO SOSPECHOSO) ---")
resultado_2 = inferencia.query(
    variables=['Fraude'],
    evidence={'Monto_Alto': 'Sí', 'Ubicacion_Extraña': 'Sí', 'Hora_Inusual': 'Sí', 
              'Dispositivo_Nuevo': 'Sí', 'Frecuencia_Transacciones': 'Baja'}
)
print(resultado_2)

--- NUEVO SAMPLE 1 (CASO LEGAL) ---
+------------+---------------+
| Fraude     |   phi(Fraude) |
+============+===============+
| Fraude(No) |        0.9741 |
+------------+---------------+
| Fraude(Sí) |        0.0259 |
+------------+---------------+

--- NUEVO SAMPLE 2 (CASO SOSPECHOSO) ---
+------------+---------------+
| Fraude     |   phi(Fraude) |
+============+===============+
| Fraude(No) |        0.2517 |
+------------+---------------+
| Fraude(Sí) |        0.7483 |
+------------+---------------+
